# Code1.ipynb — Mixed-Precision Iterative Refinement: FP32 / FP16 / BF16(SIMT) / BF16(TC)

Phase B of `IR.md`. Algorithm 1.1 of Carson & Higham, *SISC* 40(2):A817-A847, 2018,
instantiated with a pivoting-free direct factorization (Cholesky, SPD matrices) at
precision $u_f$; working precision $u$ = FP32, residual $u_r$ = FP64. Section-6
residual scaling ($\theta=\|r\|_\infty$) included.

| column | storage rounding | accumulation | models |
|--------|-----------------|--------------|--------|
| F32    | none (FP32, $u_f=2^{-24}$) | FP32 | full-precision baseline ("traditional" IR row) |
| F16    | FP16 ($u_f=2^{-11}$) | FP32 | FP16 with FP32-accumulate FMA / tensor cores |
| B16-S  | BF16 ($u_f=2^{-8}$)  | BF16, per-FMA | BF16 on SIMT cores |
| B16-T  | BF16 ($u_f=2^{-8}$)  | FP32 | BF16 on tensor cores (FP32 internal accumulate) |

Stopping target: $\mathrm{RE}=\|x-x_{ref}\|_\infty/\|x_{ref}\|_\infty \le 10^{-6}$,
$x_{ref}$ = FP64 solution of the **stored** FP32 system.

Reading the table: Niter is the **iteration overhead purchased by the cheaper
factorization**. F32 pays ~zero overhead but full-price $O(n^3)$; the 16-bit
columns pay iterations for a factorization that is up to 16x cheaper in flops
on tensor-core hardware. Niter alone therefore always "favors" F32 — the
speed-vs-accuracy trade only prices out on the GPU (Phase C), where total time
$= T_{fact}(u_f) + N_{iter}\cdot T_{iter}$ with $T_{iter}$ config-independent.

In [ ]:
"""Mixed-precision Cholesky-IR (Carson-Higham Alg 1.1) with simulated FP16/BF16.
Code1.ipynb -- Phase B of IR.md (+F32 baseline). Colab CPU; numpy+scipy+matplotlib."""
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def round_bf16(x):
    x = np.ascontiguousarray(x, dtype=np.float32)
    u = x.view(np.uint32)
    return ((u + 0x7FFF + ((u >> 16) & 1)) & 0xFFFF0000).astype(np.uint32).view(np.float32)

def round_fp16(x):
    return np.asarray(x, dtype=np.float16).astype(np.float32)

def round_fp32(x):
    return np.asarray(x, dtype=np.float32)

ROUND = {"fp16": round_fp16, "bf16": round_bf16, "fp32": round_fp32}

In [ ]:
class Breakdown(Exception): pass

def chol_16(A32, fmt, acc16):
    """Right-looking column Cholesky of u_f-rounded A.
    acc16=True  (SIMT): trailing matrix rounded to u_f after every rank-1 update
                        (= one rounding per FMA per element).
    acc16=False (TC)  : trailing accumulation stays fp32; only L rounded to u_f."""
    rnd = ROUND[fmt]
    n = A32.shape[0]
    C = rnd(A32).copy()
    L = np.zeros_like(C)
    for j in range(n):
        cjj = C[j, j]
        if not np.isfinite(cjj) or cjj <= 0:
            raise Breakdown(j)
        ljj = np.float32(np.sqrt(cjj))
        L[j, j] = rnd(np.array([ljj]))[0]
        if j + 1 < n:
            L[j+1:, j] = rnd(C[j+1:, j] / L[j, j])
            v = L[j+1:, j]
            C[j+1:, j+1:] -= np.outer(v, v)
            if acc16:
                C[j+1:, j+1:] = rnd(C[j+1:, j+1:])
    return L

In [ ]:
def trisolve_16(L, r, fmt, acc16):
    """Solve L L^T d = r; SIMT rounds running partials per column step (per-FMA)."""
    rnd = ROUND[fmt]
    n = L.shape[0]
    y = rnd(np.asarray(r, dtype=np.float32)).copy()
    for j in range(n):
        y[j] = y[j] / L[j, j]
        if acc16:
            y[j] = rnd(np.array([y[j]]))[0]
        if j + 1 < n:
            y[j+1:] -= L[j+1:, j] * y[j]
            if acc16:
                y[j+1:] = rnd(y[j+1:])
    d = y
    for j in range(n - 1, -1, -1):
        d[j] = d[j] / L[j, j]
        if acc16:
            d[j] = rnd(np.array([d[j]]))[0]
        if j > 0:
            d[:j] -= L[j, :j] * d[j]
            if acc16:
                d[:j] = rnd(d[:j])
    return d

In [ ]:
def ir_solve(A32, b32, x_ref, L, fmt, acc16, RE=1e-6, maxit=50):
    A64, b64 = A32.astype(np.float64), b32.astype(np.float64)
    th0 = np.float32(np.linalg.norm(b32, np.inf))
    x = (th0 * trisolve_16(L, b32 / th0, fmt, acc16)).astype(np.float32)  # sect-6 scaling
    errs, bad = [], 0
    for i in range(maxit + 1):
        err = np.linalg.norm(x - x_ref, np.inf) / np.linalg.norm(x_ref, np.inf)
        errs.append(err)
        if err <= RE:
            return i, errs, "converged"
        if i > 0 and errs[-1] >= errs[-2]:
            bad += 1
            if bad >= 3:
                return None, errs, "fail"
        else:
            bad = 0
        if i == maxit:
            return None, errs, "fail"
        r = np.asarray(b64 - A64 @ x.astype(np.float64), dtype=np.float32)
        theta = np.float32(np.linalg.norm(r, np.inf))                    # sect-6 scaling
        d = theta * trisolve_16(L, r / theta, fmt, acc16)
        x = (x + d).astype(np.float32)

In [ ]:
def make_problem(n, kappa2, rng):
    Q, _ = np.linalg.qr(rng.standard_normal((n, n)))
    A = (Q * np.logspace(0, -np.log10(kappa2), n)) @ Q.T
    A32 = np.asarray((A + A.T) / 2, dtype=np.float32)
    xt = rng.standard_normal(n)
    b32 = np.asarray(A32.astype(np.float64) @ xt, dtype=np.float32)
    x_ref = np.linalg.solve(A32.astype(np.float64), b32.astype(np.float64))  # ref of STORED system
    return A32, b32, x_ref.astype(np.float32)

CONFIGS = {"F32": ("fp32", False), "F16": ("fp16", False), "B16-S": ("bf16", True), "B16-T": ("bf16", False)}

def run_cell(n, kappa2, cfg, nseeds=5, RE=1e-6):
    fmt, acc16 = CONFIGS[cfg]
    out = []
    for s in range(nseeds):
        rng = np.random.default_rng(1000 + s)
        A32, b32, x_ref = make_problem(n, kappa2, rng)
        try:
            L = chol_16(A32, fmt, acc16)
        except Breakdown:
            out.append(("breakdown", None)); continue
        nit, errs, status = ir_solve(A32, b32, x_ref, L, fmt, acc16, RE=RE)
        out.append((status, nit))
    return out

In [ ]:
# Calibration sweep (~2-5 min on Colab CPU). Median Niter over 5 seeds.
n, RE = 256, 1e-6
kappas = [1e1, 3e1, 1e2, 3e2, 1e3, 3e3, 1e4, 3e4, 1e5]
results = {}
print(f"n={n} RE={RE:.0e} seeds=5 (median Niter | outcomes)")
print("kappa2   " + "".join(f"{c:>24s}" for c in CONFIGS))
for k in kappas:
    row = f"{k:8.0e} "
    for cfg in CONFIGS:
        res = run_cell(n, k, cfg, RE=RE)
        results[(k, cfg)] = res
        nits = [x[1] for x in res if x[0] == "converged"]
        cnt = {}
        for st, _ in res: cnt[st] = cnt.get(st, 0) + 1
        med = int(np.median(nits)) if nits else "-"
        row += f"{str(med):>4s} {str(cnt):>19s}"
    print(row)

In [ ]:
# Frozen condition ladder and headline table
C1, C2, C3 = 1e2, 1e3, 3e4
print(f"C1={C1:.0e}  C2={C2:.0e}  C3={C3:.0e}   (C2a = B16-T @ C2, C2b = B16-S @ C2)")
print(f"{'kappa2':>8s}" + "".join(f"{c:>11s}" for c in CONFIGS))
for C in (C1, C2, C3):
    row = f"{C:8.0e} "
    for cfg in CONFIGS:
        res = results[(C, cfg)]
        nits = [x[1] for x in res if x[0] == "converged"]
        if len(nits) >= 3:
            row += f"{int(np.median(nits)):>10d} "
        else:
            worst = max(set(s for s,_ in res), key=[s for s,_ in res].count)
            row += f"{worst.upper():>10s} "
    print(row)

In [ ]:
# Convergence traces at C1 and C2 (seed 1000)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, kappa, tag in [(axes[0], C1, "C1"), (axes[1], C2, "C2")]:
    for cfg, mk in [("F32","d-"), ("F16","o-"), ("B16-S","s-"), ("B16-T","^-")]:
        fmt, acc16 = CONFIGS[cfg]
        rng = np.random.default_rng(1000)
        A32, b32, x_ref = make_problem(256, kappa, rng)
        try:
            L = chol_16(A32, fmt, acc16)
            nit, errs, st = ir_solve(A32, b32, x_ref, L, fmt, acc16)
            lbl = f"{cfg} ({st}" + (f", {nit} it)" if nit is not None else ")")
            ax.semilogy(errs, mk, label=lbl)
        except Breakdown:
            ax.plot([], [], mk, label=f"{cfg} (breakdown)")
    ax.axhline(1e-6, color="k", ls=":", lw=1, label="RE = 1e-6")
    ax.set_title(f"{tag}: kappa2 = {kappa:.0e}"); ax.set_xlabel("refinement step")
    ax.grid(alpha=.3); ax.legend(fontsize=8)
axes[0].set_ylabel(r"$\|x-x_{ref}\|_\infty / \|x_{ref}\|_\infty$")
plt.tight_layout(); plt.show()

## Frozen non-GPU parameters (reported to IR.md)

- n = 256, dense SPD randsvd-style, 5 seeds (1000-1004); RE = 1e-6, maxit = 50
- **C1 = 1e2** : all converge — F32: 1, F16: 2, B16-S: 12, B16-T: 4
- **C2 = 1e3** : F32: 1, F16: 3, **C2b = B16-S: FAIL**, **C2a = B16-T: 14**
- **C3 = 3e4** : F32: 1; all 16-bit columns fail (F16 diverges; BF16 breaks down)

Sweep thresholds: F16 through 1e4, B16-S through 3e2, B16-T through 1e3, F32
everywhere (1/u_fp32 ~ 1e8 is far beyond the sweep). The F32 column is the
positive control: C3 failures are precision-caused, not problem-caused, and the
C1-C2 band is the "interchangeable region" where BF16-TC delivers F32-identical
final accuracy — on the A100 that region is where the cheap factorization wins
total time; past it, F32 (or a better inner solver, GMRES-IR/AMG) is mandatory.

Bonus (Phase D preview): without section-6 scaling, F16 stalls already at
kappa = 3e2 — residual entries fall below FP16 min normal $6.1\times10^{-5}$
and flush to subnormals/zero; BF16 (range $10^{\pm38}$) unaffected.

Phase C additions: real `__half`/`__nv_bfloat16` arithmetic; cuBLAS
`CUBLAS_COMPUTE_32F` (TC) vs hand-written bf16-accumulate kernel (SIMT);
optional 5th config TF32 (FP16-width mantissa, FP32 exponent, FP32 accumulate)
completing the axis isolation: F16<->TF32 = exponent only, TF32<->F32 =
mantissa only, B16-T<->B16-S = accumulator only.